In [6]:
# Imports.

from concurrent.futures import ThreadPoolExecutor
import gc
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, set_seed
from trl import SFTConfig, SFTTrainer
from tqdm.auto import tqdm

import os
os.environ["CUDA_VISIBLE_DEVICES"] = '3'

In [7]:
# Constants.

DATA_CSV = Path("artifacts/processed_data/chexpertplus_frontal_5labels.csv")
BENCHMARK_DIR = Path("temp/lora_sft_benchmark")
MEDGEMMA_MODEL_ID = "google/medgemma-4b-it"
MODEL_DTYPE = torch.bfloat16
RANDOM_STATE = 42

TARGET_LABELS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
PROMPT_ORDERS = ["image_first", "text_first"]
BATCH_SIZES = [32, 48, 64]
EFFECTIVE_BATCH_SIZE = 128
BENCHMARK_STUDIES = 512
BENCHMARK_STEPS = 10
LEARNING_RATE = 1e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
IMAGE_LOAD_NUM_WORKERS = 4
IMAGE_PROCESS_BATCH_SIZE = 24
DATALOADER_NUM_WORKERS = 4
DECODER_LINEAR_NAMES = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}

BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
set_seed(RANDOM_STATE)


In [8]:
# Load a benchmark dataset and expand each study into prompt-condition examples.

df = pd.read_csv(DATA_CSV)
train_df = df[df["probe_split"] == "train"].head(BENCHMARK_STUDIES).reset_index(drop=True)

rows = []
for image_idx, row in train_df.iterrows():
    for prompt_order in PROMPT_ORDERS:
        for label in TARGET_LABELS:
            rows.append({
                "image_idx": image_idx,
                "image_path": row["image_path"],
                "prompt_order": prompt_order,
                "finding": label,
                "answer": "yes" if row[label] == 1 else "no",
            })

train_dataset = Dataset.from_pandas(pd.DataFrame(rows), preserve_index=False)
print("benchmark studies", len(train_df))
print("benchmark examples", len(train_dataset))


benchmark studies 512
benchmark examples 5120


In [9]:
# Processor, prompt format, and collators.

processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token


def prompt_text(prompt_order, label):
    question = f"Question: Is there {label.lower()} in this image? Answer yes or no."
    image_item = {"type": "image"}
    if prompt_order == "image_first":
        content = [image_item, {"type": "text", "text": f"\n{question}\nAnswer: "}]
    else:
        content = [{"type": "text", "text": f"{question}\n"}, image_item, {"type": "text", "text": "\nAnswer: "}]
    text = processor.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=False, tokenize=False)
    if isinstance(text, list):
        text = text[0]
    return text.replace(processor.boi_token, processor.full_image_sequence)


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def collate_text(examples):
    input_rows = []
    label_rows = []
    for example in examples:
        prompt_ids = processor.tokenizer(prompt_text(example["prompt_order"], example["finding"])).input_ids
        answer_ids = processor.tokenizer(example["answer"], add_special_tokens=False).input_ids
        input_rows.append(prompt_ids + answer_ids)
        label_rows.append([-100] * len(prompt_ids) + answer_ids)

    max_len = max(len(row) for row in input_rows)
    input_ids = torch.full((len(examples), max_len), processor.tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(examples), max_len), dtype=torch.long)
    labels = torch.full((len(examples), max_len), -100, dtype=torch.long)

    for i, (input_row, label_row) in enumerate(zip(input_rows, label_rows)):
        input_ids[i, :len(input_row)] = torch.tensor(input_row)
        attention_mask[i, :len(input_row)] = 1
        labels[i, :len(label_row)] = torch.tensor(label_row)

    batch = {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}
    if hasattr(processor, "create_mm_token_type_ids"):
        token_type_ids = processor.create_mm_token_type_ids(input_ids)
        batch["token_type_ids"] = token_type_ids if torch.is_tensor(token_type_ids) else torch.tensor(token_type_ids)
    return batch


def collate_on_the_fly(examples):
    batch = collate_text(examples)
    images = [load_rgb(example["image_path"]) for example in examples]
    batch["pixel_values"] = processor.image_processor(images=images, return_tensors="pt", do_pan_and_scan=False)["pixel_values"].to(MODEL_DTYPE)
    return batch


def collate_cached_pixel_values(examples):
    batch = collate_text(examples)
    batch["pixel_values"] = cached_pixel_values[[int(example["image_idx"]) for example in examples]]
    return batch

sample_batch = collate_on_the_fly([train_dataset[0], train_dataset[1]])
print("yes ids", processor.tokenizer("yes", add_special_tokens=False).input_ids)
print("no ids", processor.tokenizer("no", add_special_tokens=False).input_ids)
print("supervised tokens", sample_batch["labels"].ne(-100).sum(dim=1).tolist())
del sample_batch


yes ids [4443]
no ids [1904]
supervised tokens [1, 1]


In [10]:
# Cache processed image tensors once for the cached-pixel benchmark.

cache_start = time.perf_counter()
with ThreadPoolExecutor(max_workers=IMAGE_LOAD_NUM_WORKERS) as pool:
    cached_images = list(tqdm(pool.map(load_rgb, train_df["image_path"]), total=len(train_df), desc="Load images"))
image_load_sec = time.perf_counter() - cache_start

process_start = time.perf_counter()
pixel_chunks = []
for start in tqdm(range(0, len(cached_images), IMAGE_PROCESS_BATCH_SIZE), desc="Process images"):
    end = min(start + IMAGE_PROCESS_BATCH_SIZE, len(cached_images))
    pixel_inputs = processor.image_processor(images=cached_images[start:end], return_tensors="pt", do_pan_and_scan=False)
    pixel_chunks.append(pixel_inputs["pixel_values"].to(dtype=MODEL_DTYPE, device="cpu"))
cached_pixel_values = torch.cat(pixel_chunks, dim=0)
image_process_sec = time.perf_counter() - process_start

del cached_images, pixel_chunks
print("image_load_sec", round(image_load_sec, 2))
print("image_process_sec", round(image_process_sec, 2))
print("cached_pixel_values", tuple(cached_pixel_values.shape), cached_pixel_values.dtype)


Process images: 100%|██████████| 22/22 [00:24<00:00,  1.13s/it]


image_load_sec 8.79
image_process_sec 26.12
cached_pixel_values (512, 3, 896, 896) torch.bfloat16


In [11]:
# Benchmark current on-the-fly collator vs cached pixel_values collator.

benchmark_rows = []
collators = [
    ("on_the_fly", collate_on_the_fly, 0.0),
    ("cached_pixel_values", collate_cached_pixel_values, image_load_sec + image_process_sec),
]

for collator_name, collator, setup_sec in collators:
    for batch_size in BATCH_SIZES:
        print(collator_name, "batch_size", batch_size)
        model = AutoModelForImageTextToText.from_pretrained(MEDGEMMA_MODEL_ID, dtype=MODEL_DTYPE)
        model.config.use_cache = False

        target_modules = [
            name for name, module in model.named_modules()
            if isinstance(module, torch.nn.Linear)
            and "language_model" in name
            and "vision_tower" not in name
            and "multi_modal_projector" not in name
            and name.split(".")[-1] in DECODER_LINEAR_NAMES
        ]

        trainer = SFTTrainer(
            model=model,
            args=SFTConfig(
                output_dir=str(BENCHMARK_DIR / f"{collator_name}_batch_{batch_size}"),
                per_device_train_batch_size=batch_size,
                gradient_accumulation_steps=math.ceil(EFFECTIVE_BATCH_SIZE / batch_size),
                max_steps=BENCHMARK_STEPS,
                learning_rate=LEARNING_RATE,
                bf16=True,
                max_length=None,
                packing=False,
                warmup_ratio=0.03,
                lr_scheduler_type="cosine",
                max_grad_norm=1.0,
                logging_steps=1,
                save_strategy="no",
                report_to="none",
                remove_unused_columns=False,
                dataloader_num_workers=DATALOADER_NUM_WORKERS,
                dataset_kwargs={"skip_prepare_dataset": True},
                gradient_checkpointing=False,
                seed=RANDOM_STATE,
            ),
            train_dataset=train_dataset,
            data_collator=collator,
            processing_class=processor,
            peft_config=LoraConfig(
                r=LORA_R,
                lora_alpha=LORA_ALPHA,
                lora_dropout=LORA_DROPOUT,
                bias="none",
                task_type="CAUSAL_LM",
                target_modules=target_modules,
            ),
        )

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        start_time = time.perf_counter()
        try:
            trainer.train()
            ok = True
            error = ""
        except RuntimeError as exc:
            ok = False
            error = str(exc).split("\n")[0]
            if "out of memory" not in error.lower():
                raise
        train_sec = time.perf_counter() - start_time
        peak_gb = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else np.nan

        benchmark_rows.append({
            "collator": collator_name,
            "batch_size": batch_size,
            "gradient_accumulation_steps": math.ceil(EFFECTIVE_BATCH_SIZE / batch_size),
            "ok": ok,
            "setup_sec": setup_sec,
            "train_sec": train_sec,
            "total_sec_with_setup": setup_sec + train_sec,
            "optimizer_steps": BENCHMARK_STEPS,
            "effective_examples_per_sec": (EFFECTIVE_BATCH_SIZE * BENCHMARK_STEPS) / train_sec if ok else np.nan,
            "peak_gpu_gb": peak_gb,
            "error": error,
        })

        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()

benchmark_df = pd.DataFrame(benchmark_rows)
display(benchmark_df.sort_values(["collator", "batch_size"]))


on_the_fly batch_size 32


Loading weights: 100%|██████████| 883/883 [00:00<00:00, 1945.54it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
1,12.703938
2,13.337395
3,5.382338
4,1.084771
5,0.919046
6,0.634976
7,0.683699
8,0.613840
9,0.608889
10,0.499932


on_the_fly batch_size 48


Loading weights: 100%|██████████| 883/883 [00:00<00:00, 939.50it/s] 
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


...

cached_pixel_values batch_size 48


Loading weights: 100%|██████████| 883/883 [00:01<00:00, 818.98it/s] 
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


cached_pixel_values batch_size 64


Loading weights: 100%|██████████| 883/883 [00:01<00:00, 871.57it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


,collator,batch_size,gradient_accumulation_steps,ok,setup_sec,train_sec,total_sec_with_setup,optimizer_steps,effective_examples_per_sec,peak_gpu_gb,error
3,cached_pixel_values,32,4,True,34.914741,69.215576,104.130317,10,18.492947,104.929350,
4,cached_pixel_values,48,3,False,34.914741,3.724340,38.639081,10,NaN,125.778550,CUDA out of memory. Tried to allocate 13.31 Gi...
5,cached_pixel_values,64,2,False,34.914741,4.710604,39.625346,10,NaN,130.849732,CUDA out of memory. Tried to allocate 8.94 GiB...
0,on_the_fly,32,4,True,0.000000,78.235833,78.235833,10,16.360790,104.929003,
1,on_the_fly,48,3,False,0.000000,14.123160,14.123160,10,NaN,125.777493,CUDA out of memory. Tried to allocate 13.31 Gi...
2,on_the_fly,64,2,False,0.000000,17.547337,17.547337,10,NaN,130.851808,CUDA out of memory. Tried to allocate 8.94 GiB...
